# Notebook 3: Ràng buộc, biến slack và hằng số phạt

Đi kèm [Phần A, Mục 4.3](../../docs/A-translation/04-problem-transformation.md): Ví dụ 13 và 14, các phương trình (32)–(38). Notebook kiểm tra bằng duyệt vét cạn rằng hàm đã phạt có cùng nghiệm tối ưu với bài toán có ràng buộc, đồng thời khảo sát hai nhận xét của người dịch: cận dưới của $h$ khá lỏng, và $P = 127$ lớn hơn mức cần thiết.

In [1]:
import itertools, sys
import sympy as sp
sys.path.insert(0, "../../code")
from qco_utils import (multilinear, evaluate, brute_force, lower_bound, upper_bound,
                       penalty_constant, integer_encoding)

## 1. Ví dụ 13: biến slack cho ràng buộc $3x_1x_4x_6 - 4x_3x_5 \le 0$

In [2]:
x1, x3, x4, x5, x6 = xs = sp.symbols("x1 x3 x4 x5 x6")
g = multilinear(3*x1*x4*x6 - 4*x3*x5, xs)
gmin, _ = brute_force(g, 5)
print("min g =", gmin, "=> s thuộc [0, %d]" % (-gmin))
a, w = integer_encoding(0, -gmin)
print("trọng số của slack:", w, "(bài báo: s = t1 + 2 t2 + t3)")

min g = -4 => s thuộc [0, 4]
trọng số của slack: [1, 2, 1] (bài báo: s = t1 + 2 t2 + t3)


## 2. Ví dụ 14: ràng buộc $(z-1)^2 \le 6$ cho hàm của Ví dụ 10

Các biến: $z = -3 + x_1 + 2x_2 + 3x_3$, $f(z) = z^3 - 6z$, $h(z) = (z-1)^2 - 6$.

In [3]:
x1, x2, x3 = X = sp.symbols("x1 x2 x3")
z = -3 + x1 + 2*x2 + 3*x3
f_poly = multilinear(z**3 - 6*z, X)
h_poly = multilinear((z - 1)**2 - 6, X)

paper37 = {frozenset(): 10, frozenset({0}): -7, frozenset({1}): -12, frozenset({2}): -15,
           frozenset({0, 1}): 4, frozenset({0, 2}): 6, frozenset({1, 2}): 12}
print("h khớp phương trình (37):", h_poly == paper37)

print("lb(h) theo (34) =", lower_bound(h_poly), "| min h thực (vét cạn) =", brute_force(h_poly, 3)[0])
print("P theo (36) =", penalty_constant(f_poly), "(bài báo: 127)")
print("ub(f) =", upper_bound(f_poly), " lb(f) =", lower_bound(f_poly),
      "| khoảng dao động thực của f:", (brute_force(f_poly, 3)[0], max(evaluate(f_poly, x) for x in itertools.product((0, 1), repeat=3))))

h khớp phương trình (37): True
lb(h) theo (34) = -24 | min h thực (vét cạn) = -6
P theo (36) = 127 (bài báo: 127)
ub(f) = 63  lb(f) = -63 | khoảng dao động thực của f: (-9, 9)


Cận dưới của $h$ là $-24$ nhưng giá trị nhỏ nhất thực là $-6$. Cận này an toàn nhưng lỏng: nó cần slack trong $[0, 24]$, tức 5 biến phụ, trong khi $[0, 6]$ chỉ cần 3.

In [4]:
for hi in (24, 6):
    a, w = integer_encoding(0, hi)
    print(f"slack trong [0, {hi:2d}]: {len(w)} biến phụ, trọng số {w}")

slack trong [0, 24]: 5 biến phụ, trọng số [1, 2, 4, 8, 9]
slack trong [0,  6]: 3 biến phụ, trọng số [1, 2, 3]


## 3. Hàm đã phạt $f'(x, t) = f(x) + P\,(h(x) + s)^2$

Ta dựng $f'$ với slack 5 biến (như bài báo) và kiểm tra: cực tiểu của $f'$ trên cả 8 biến nhị phân phải bằng cực tiểu của $f$ trên các $z$ thỏa $(z-1)^2 \le 6$.

In [5]:
def build(slack_hi, P):
    a, w = integer_encoding(0, slack_hi)
    ts = sp.symbols(f"t1:{len(w)+1}")
    s = sum(wi * ti for wi, ti in zip(w, ts))
    allvars = list(X) + list(ts)
    F = multilinear(z**3 - 6*z + P*((z - 1)**2 - 6 + s)**2, allvars)
    return F, len(allvars), w

# đáp án đúng: duyệt các z thỏa ràng buộc
feasible = {zz: zz**3 - 6*zz for zz in range(-3, 4) if (zz - 1)**2 <= 6}
print("z khả thi:", feasible)
fstar = min(feasible.values())
print("giá trị tối ưu mong đợi:", fstar)

F, n, w = build(24, 127)
best, arg = brute_force(F, n)
print("min f' =", best, "| nghiệm tối ưu (x1,x2,x3,t...):", arg)
zs = {-3 + x[0] + 2*x[1] + 3*x[2] for x in arg}
print("các z tương ứng:", zs)
assert best == fstar and all((zz - 1)**2 <= 6 for zz in zs)

z khả thi: {-1: 5, 0: 0, 1: -5, 2: -4, 3: 9}
giá trị tối ưu mong đợi: -5
min f' = -5 | nghiệm tối ưu (x1,x2,x3,t...): [(1, 0, 1, 0, 1, 1, 0, 0)]
các z tương ứng: {1}


## 4. $P = 127$ có cần thiết không?

Với $f$ và $h$ trên, ta tìm $P$ nhỏ nhất sao cho mọi nghiệm tối ưu của $f'$ đều khả thi và đạt giá trị tối ưu của bài toán gốc. Phép thử được thực hiện cho slack 5 biến (cận lỏng) và slack 3 biến (cận chặt).

In [6]:
def smallest_working_P(slack_hi):
    a, w = integer_encoding(0, slack_hi)
    ts = sp.symbols(f"t1:{len(w)+1}")
    s = sum(wi * ti for wi, ti in zip(w, ts))
    allvars = list(X) + list(ts)
    fpoly = multilinear(z**3 - 6*z, allvars)
    gpoly = multilinear(((z - 1)**2 - 6 + s)**2, allvars)
    combos = list(itertools.product((0, 1), repeat=len(allvars)))
    fv = [evaluate(fpoly, c) for c in combos]
    gv = [evaluate(gpoly, c) for c in combos]
    fstar = min(f for f, g in zip(fv, gv) if g == 0)
    for P in range(1, 200):
        vals = [f + P * g for f, g in zip(fv, gv)]
        m = min(vals)
        if all(g == 0 and f == fstar for f, g, v in zip(fv, gv, vals) if v == m):
            return P, len(w)

for hi in (24, 6):
    P, nb_ = smallest_working_P(hi)
    print(f"slack trong [0, {hi:2d}] ({nb_} biến phụ): P nhỏ nhất dùng được = {P}")

slack trong [0, 24] (5 biến phụ): P nhỏ nhất dùng được = 1
slack trong [0,  6] (3 biến phụ): P nhỏ nhất dùng được = 1


Công thức (36) cho $P = 127$ vì nó chỉ dựa vào tổng các hệ số tuyệt đối, không dùng thông tin gì khác về bài toán. Giá trị này an toàn nhưng lớn hơn rất nhiều so với mức tối thiểu của thể hiện cụ thể này.

Cần lưu ý đừng suy ra rằng $P = 1$ luôn đủ. Ở thể hiện này mọi lời giải vi phạm ràng buộc đều vi phạm khá nhiều (với $z = -3$ thì $h = 10$, với $z = -2$ thì $h = 3$), và vì phạt dùng bình phương nên $(h+s)^2 \ge 9$ đã vượt xa khoảng dao động của $f$. Với một ràng buộc mà mức vi phạm nhỏ nhất chỉ là 1, ta sẽ cần $P$ lớn hơn. Công thức (36) là cận an toàn cho mọi trường hợp, còn một cận chặt hơn thì phải khai thác cấu trúc của bài toán.

Điều này quan trọng trong thực tế: hằng số phạt lớn làm hàm mục tiêu có dải giá trị rộng, và trên phần cứng lượng tử (độ phân giải hữu hạn của các hệ số) có thể làm "chìm" phần mục tiêu thật. Đây là một lý do để chọn $P$ cẩn thận, và là chủ đề của Phần B.